# ForecastMonitor: Tracking Performance and Alerting

In production forecasting, model accuracy degrades over time due to structural breaks,
regime changes, or data drift. A monitoring system that tracks performance and triggers
alerts is essential for maintaining forecast quality.

**ForecastMonitor** provides:
- Rolling and cumulative accuracy tracking (RMSE, MAE, MAPE)
- Bias evolution monitoring
- Forecast vs actual comparison with prediction intervals
- Degradation detection (recent vs historical accuracy)

**AlertSystem** adds rule-based alerting:
- Preset rules: RMSE spikes, bias drift, coverage drops, model changes
- Custom rules with configurable thresholds, windows, and severity levels
- Alert history and summary reporting

This notebook demonstrates the full monitoring lifecycle: from setting up a monitor,
tracking forecasts over time, detecting degradation, configuring alerts, and implementing
automated re-training responses.

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from forecastbox.pipeline import (
    ForecastPipeline,
    ForecastMonitor,
    RecurringForecast,
    AlertSystem,
    AlertRule,
    Alert,
)

# Add helpers path
sys.path.insert(0, "../utils")
from helpers import load_macro_brazil

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("forecastbox monitoring modules loaded successfully.")

## 1. Setting Up the Monitor

The `ForecastMonitor` is attached to a `ForecastPipeline` and tracks pairs of
forecasted vs actual values over time. We set it up by:

1. Creating a pipeline for our target variable
2. Instantiating a `ForecastMonitor` linked to the pipeline
3. Populating it with historical forecast-actual pairs

In [ ]:
# Load macro_brazil dataset
df = load_macro_brazil()
print(f"Dataset: {df.shape[0]} months, columns: {list(df.columns)}")

# Create a pipeline for IPCA inflation forecasting
pipeline = ForecastPipeline(
    data_source=df,
    target="ipca",
    models=["auto_arima", "auto_ets", "naive"],
    combination="mean",
    evaluation=["rmse", "mae"],
    horizon=12,
    preprocess=["missing_fill"],
)

# Set up the monitor
monitor = ForecastMonitor(pipeline=pipeline)
print(f"\nMonitor created: {monitor}")
print(f"Actuals tracked: {len(monitor.actuals)}")
print(f"Forecasts tracked: {len(monitor.forecasted)}")

## 2. Tracking Forecast Accuracy Over Time

We simulate 12 months of one-step-ahead forecasting. At each month:
1. Use data up to month $t$ to fit models and forecast month $t+1$
2. When month $t+1$ is realized, record both forecast and actual
3. Track rolling accuracy metrics over time

In [ ]:
# Simulate 12 months of forecasts with progressive data arrival
# Hold out last 24 months, use first 12 for stable period, last 12 for degradation
n_holdout = 24
ipca = df["ipca"]
rng = np.random.default_rng(42)

for i in range(n_holdout):
    train_end = len(ipca) - n_holdout + i
    train_data = ipca.iloc[:train_end]
    
    # Create a simple pipeline for this month
    month_pipeline = ForecastPipeline(
        data_source=train_data.to_frame(),
        target="ipca",
        models=["auto_arima"],
        horizon=1,
        preprocess=["missing_fill"],
    )
    result = month_pipeline.run()
    
    # Get the 1-step-ahead forecast
    fc = list(result.forecasts.values())[0]
    forecast_date = ipca.index[train_end]
    actual_value = float(ipca.iloc[train_end])
    forecast_point = float(fc.point[0])
    
    # For the last 6 months, inject degradation (simulating structural break)
    if i >= 18:
        forecast_point += rng.normal(0.3, 0.15)  # systematic bias + noise
    
    lower_95 = forecast_point - 1.96 * float(fc.point.std()) if fc.lower_95 is not None else forecast_point - 0.3
    upper_95 = forecast_point + 1.96 * float(fc.point.std()) if fc.upper_95 is not None else forecast_point + 0.3
    
    # Record in monitor
    monitor.add_actual(forecast_date, actual_value)
    monitor.add_forecast(forecast_date, forecast_point, lower_95, upper_95)

print(f"Monitor now tracks {len(monitor.actuals)} actuals and {len(monitor.forecasted)} forecasts")

# Generate accuracy report
report = monitor.accuracy_report()
print(f"\n{report.summary()}")

## 3. Detecting Performance Degradation

Degradation detection compares recent forecast accuracy to historical accuracy.
If recent RMSE exceeds historical RMSE by more than a threshold (default: 1.5x),
degradation is flagged.

We also visualize rolling metrics over time to spot trends and structural breaks.

In [ ]:
# Test for degradation with different thresholds
for threshold in [1.2, 1.5, 2.0]:
    degraded = monitor.degradation_test(window=6, threshold=threshold)
    status = "DEGRADED" if degraded else "OK"
    print(f"Threshold {threshold}x: {status}")

# Rolling accuracy metrics
rolling_rmse = monitor.rolling_accuracy(window=6, metric="rmse")
rolling_mae = monitor.rolling_accuracy(window=6, metric="mae")
cumulative_rmse = monitor.cumulative_accuracy(metric="rmse")

# Bias tracker
bias = monitor.bias_tracker()

# Plot metrics evolution with degradation zone
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Rolling RMSE
ax = axes[0, 0]
ax.plot(rolling_rmse.index, rolling_rmse.values, "b-", linewidth=2, label="Rolling RMSE (w=6)")
ax.plot(cumulative_rmse.index, cumulative_rmse.values, "b--", linewidth=1.2, label="Cumulative RMSE")
# Mark degradation zone (last 6 months)
if len(rolling_rmse) >= 6:
    ax.axvspan(rolling_rmse.index[-6], rolling_rmse.index[-1], alpha=0.15, color="red", label="Degradation zone")
ax.set_title("RMSE Evolution", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Rolling MAE
ax = axes[0, 1]
ax.plot(rolling_mae.index, rolling_mae.values, "g-", linewidth=2, label="Rolling MAE (w=6)")
ax.set_title("MAE Evolution", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Bias evolution
ax = axes[1, 0]
ax.plot(bias.index, bias.values, "darkorange", linewidth=2, label="Cumulative Bias (MFE)")
ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_title("Bias (Mean Forecast Error) Evolution", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Forecast vs Actual
monitor.plot_forecast_vs_actual(ax=axes[1, 1])

plt.suptitle("Performance Degradation Detection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Alert System

The `AlertSystem` provides rule-based alerting on top of a `ForecastMonitor`.
Rules can be defined manually or loaded from presets:

| Preset | Metric | Condition | Threshold | Severity |
|--------|--------|-----------|-----------|----------|
| `rmse_spike` | RMSE | above | 1.5x historical | warning |
| `bias_drift` | Bias | above | 0.5 | warning |
| `coverage_drop` | Hit rate | below | 80% | critical |
| `model_change` | RMSE | change | 30% | info |

In [ ]:
# Create an AlertSystem linked to the monitor
alert_system = AlertSystem(monitor=monitor)

# Add preset rules
alert_system.add_preset("rmse_spike")
alert_system.add_preset("bias_drift")
alert_system.add_preset("coverage_drop")

# Add a custom rule: MAE exceeds 0.3
alert_system.add_rule(
    name="high_mae",
    metric="mae",
    condition="above",
    threshold=1.5,
    window=6,
    severity="warning",
)

print(alert_system.summary())

# Check for triggered alerts
alerts = alert_system.check()

print(f"\n{'=' * 50}")
print(f"TRIGGERED ALERTS: {len(alerts)}")
print(f"{'=' * 50}")

for alert in alerts:
    severity_icon = {"info": "INFO", "warning": "WARN", "critical": "CRIT"}
    print(f"\n  [{severity_icon.get(alert.severity, '?')}] {alert.rule}")
    print(f"    Metric value: {alert.metric_value:.4f}")
    print(f"    Threshold:    {alert.threshold}")
    print(f"    Message:      {alert.message}")

## 5. Model Comparison Dashboard

A visual dashboard summarizing model performance: rolling metrics, model ranking,
forecast vs actual comparison, and alert timeline.

In [ ]:
# --- Model Comparison Dashboard ---
# Simulate monitoring for multiple models to compare them

model_names = ["auto_arima", "auto_ets", "naive"]
model_monitors = {}

for model_name in model_names:
    m = ForecastMonitor(pipeline=pipeline)
    model_rng = np.random.default_rng(42 + hash(model_name) % 1000)
    
    for i in range(n_holdout):
        train_end = len(ipca) - n_holdout + i
        actual_val = float(ipca.iloc[train_end])
        forecast_date = ipca.index[train_end]
        
        # Simulate model-specific forecasts with different error profiles
        base_forecast = actual_val + model_rng.normal(0, 0.08)
        if model_name == "naive":
            base_forecast = float(ipca.iloc[train_end - 1])  # naive = last value
        elif model_name == "auto_ets":
            base_forecast = actual_val + model_rng.normal(0.02, 0.1)
        
        # Inject degradation in last 6 months for all models
        if i >= 18:
            base_forecast += model_rng.normal(0.15, 0.1)
        
        m.add_actual(forecast_date, actual_val)
        m.add_forecast(forecast_date, base_forecast,
                       base_forecast - 0.3, base_forecast + 0.3)
    
    model_monitors[model_name] = m

# Dashboard: 2x2 layout
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Rolling RMSE comparison
ax = axes[0, 0]
for name, m in model_monitors.items():
    rolling = m.rolling_accuracy(window=6, metric="rmse")
    if not rolling.empty:
        ax.plot(rolling.index, rolling.values, linewidth=2, label=name)
ax.set_title("Rolling RMSE by Model (window=6)", fontsize=12, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Model ranking (overall metrics)
ax = axes[0, 1]
ranking_data = []
for name, m in model_monitors.items():
    report = m.accuracy_report()
    ranking_data.append({
        "model": name,
        "RMSE": report.overall_metrics.get("rmse", np.nan),
        "MAE": report.overall_metrics.get("mae", np.nan),
        "Bias": abs(report.overall_metrics.get("mfe", 0)),
    })
ranking_df = pd.DataFrame(ranking_data).set_index("model")
ranking_df.plot(kind="bar", ax=ax, color=["steelblue", "darkorange", "forestgreen"])
ax.set_title("Overall Metrics by Model", fontsize=12, fontweight="bold")
ax.set_ylabel("Metric Value")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis="y")
ax.tick_params(axis="x", rotation=0)

# Panel 3: Forecast vs Actual (best model)
best_model_name = ranking_df["RMSE"].idxmin()
model_monitors[best_model_name].plot_forecast_vs_actual(ax=axes[1, 0])
axes[1, 0].set_title(f"Forecast vs Actual ({best_model_name})", fontsize=12, fontweight="bold")

# Panel 4: Alert timeline
ax = axes[1, 1]
alert_history = alert_system.history()
if alert_history:
    severities = {"info": 0, "warning": 1, "critical": 2}
    colors_map = {"info": "steelblue", "warning": "orange", "critical": "red"}
    for alert in alert_history:
        ax.scatter(alert.timestamp, severities.get(alert.severity, 0),
                   color=colors_map.get(alert.severity, "gray"), s=150, zorder=5,
                   edgecolors="black", linewidth=0.5)
        ax.annotate(alert.rule, (alert.timestamp, severities.get(alert.severity, 0)),
                    textcoords="offset points", xytext=(5, 10), fontsize=8)
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(["Info", "Warning", "Critical"])
    ax.set_title("Alert Timeline", fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No alerts triggered", ha="center", va="center",
            transform=ax.transAxes, fontsize=14)
    ax.set_title("Alert Timeline", fontsize=12, fontweight="bold")

plt.suptitle("Forecast Monitoring Dashboard", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# Print ranking
print("\nModel Ranking:")
display(ranking_df.sort_values("RMSE").style.highlight_min(axis=0, color="lightgreen"))

## 6. Automated Response

When degradation is detected, we can automatically re-train the pipeline with
fresh data. This implements a simple automated response loop:

1. Check alerts
2. If critical/warning alerts are triggered, re-train the pipeline
3. Update the monitor with new forecasts
4. Verify improvement

In [ ]:
# Automated re-training loop
# Simulate: if degradation detected, re-train pipeline on latest data

def automated_retrain(
    pipeline: ForecastPipeline,
    monitor: ForecastMonitor,
    alert_system: AlertSystem,
    data: pd.DataFrame,
) -> dict:
    """Check alerts and retrain pipeline if degradation detected."""
    
    result = {
        "action_taken": False,
        "alerts_before": [],
        "degraded_before": False,
        "degraded_after": False,
    }
    
    # Step 1: Check for alerts
    alerts = alert_system.check()
    result["alerts_before"] = [a.rule for a in alerts]
    
    # Step 2: Check degradation
    degraded = monitor.degradation_test(window=6, threshold=1.5)
    result["degraded_before"] = degraded
    
    if not degraded and len(alerts) == 0:
        print("No degradation detected. No action needed.")
        return result
    
    print(f"Degradation detected! {len(alerts)} alert(s) triggered.")
    print("Re-training pipeline with latest data...\n")
    
    # Step 3: Re-train pipeline with full data
    pipeline.data_source = data
    new_results = pipeline.run()
    result["action_taken"] = True
    
    print(f"Re-trained pipeline:")
    print(f"  Best model: {new_results.best_model()}")
    print(f"  Models: {list(new_results.forecasts.keys())}")
    
    # Step 4: Generate new forecast and add to monitor
    if new_results.combination is not None:
        fc = new_results.combination
    else:
        fc = list(new_results.forecasts.values())[0]
    
    # Add the new forecast (first step ahead) to monitor
    if fc.index is not None and len(fc.index) > 0:
        monitor.add_forecast(
            date=fc.index[0],
            point=float(fc.point[0]),
            lower_95=float(fc.lower_95[0]) if fc.lower_95 is not None else None,
            upper_95=float(fc.upper_95[0]) if fc.upper_95 is not None else None,
        )
        print(f"  New forecast added for {fc.index[0]}: {fc.point[0]:.4f}")
    
    return result


# Run the automated response
print("=" * 60)
print("AUTOMATED RE-TRAINING RESPONSE")
print("=" * 60)
print()

# Check current state
report_before = monitor.accuracy_report()
print(f"Current state:")
print(f"  Overall RMSE: {report_before.overall_metrics.get('rmse', 'N/A'):.4f}")
print(f"  Overall MAE:  {report_before.overall_metrics.get('mae', 'N/A'):.4f}")
print(f"  Bias:         {report_before.bias:.4f}")
print(f"  Hit rate:     {report_before.hit_rate:.1%}")
print()

# Run automated retrain
retrain_result = automated_retrain(pipeline, monitor, alert_system, df)

print(f"\n{'=' * 60}")
print(f"RESULT:")
print(f"  Action taken:     {retrain_result['action_taken']}")
print(f"  Alerts triggered: {retrain_result['alerts_before']}")
print(f"  Degraded before:  {retrain_result['degraded_before']}")
print(f"{'=' * 60}")

# Show updated alert history
print(f"\nFull alert history ({len(alert_system.history())} alerts):")
for alert in alert_system.history():
    print(f"  [{alert.severity.upper():8s}] {alert.rule}: {alert.message}")

### Exercise 1: Set up monitoring for exchange rate forecast

Load `macro_brazil.csv` and create a `ForecastPipeline` for `cambio` (BRL/USD exchange rate).
Simulate 18 months of forecasts, set up a `ForecastMonitor`, and check the accuracy report.

In [ ]:
# TODO: Exercise 1
# Hints:
# 1. Load macro_brazil with load_macro_brazil()
# 2. Create ForecastPipeline with target="cambio"
# 3. Set up ForecastMonitor(pipeline=pipeline)
# 4. Simulate 18 months: loop, add_actual() and add_forecast()
# 5. Call monitor.accuracy_report() and print the summary
# 6. Plot forecast vs actual with monitor.plot_forecast_vs_actual()

### Exercise 2: Define custom alert rules (e.g., MASE > 1.5 for 3 months)

Create an `AlertSystem` and add custom rules:
- MAE above 1.5x historical over a 3-month window (severity: warning)
- Bias above 0.3 over a 6-month window (severity: critical)
- Hit rate below 70% over a 12-month window (severity: critical)

Run `check()` and interpret the results.

In [ ]:
# TODO: Exercise 2
# Hints:
# 1. Create AlertSystem(monitor=your_monitor)
# 2. alert_system.add_rule(name="mae_spike", metric="mae", condition="above",
#                          threshold=1.5, window=3, severity="warning")
# 3. alert_system.add_rule(name="bias_alert", metric="bias", condition="above",
#                          threshold=0.3, window=6, severity="critical")
# 4. alert_system.add_rule(name="coverage_alert", metric="hit_rate", condition="below",
#                          threshold=0.7, window=12, severity="critical")
# 5. alerts = alert_system.check()
# 6. Print alert_system.summary()